# 单向与双向融合的错误比较

固定对应种子的同一 TextCNN，分别与单向 LSTM、双向 LSTM 按 0.5 / 0.5 融合。在相同验证集上比较纠正与新增错误；两个种子使用相同验证集，不能替代独立测试。


In [ ]:
import hashlib
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.lstm import LSTMClassifier
from sentiment_analysis.paths import CHECKPOINTS_DIR, CLEANED_DIR, TABLES_DIR
from sentiment_analysis.sequence_data import SentimentDataset, TOKEN_PATTERN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
validation_path = CLEANED_DIR / "val.csv"
validation_data = pd.read_csv(validation_path)
probabilities = pd.read_csv(TABLES_DIR / "textcnn_bilstm_validation_probabilities.csv")
with (TABLES_DIR / "textcnn_bilstm_ensemble_metadata.json").open(encoding="utf-8") as file:
    metadata = json.load(file)

if hashlib.sha256(validation_path.read_bytes()).hexdigest() != metadata["validation_sha256"]:
    raise ValueError("Validation data does not match the saved probabilities.")
if len(validation_data) != metadata["validation_rows"]:
    raise ValueError("Validation data size does not match the saved probabilities.")
print(f"Device: {device} | Validation: {len(validation_data):,}")


In [ ]:
# 加载单向 LSTM 检查点，核对与对应 CNN 使用相同的序列规则
checkpoint_paths = {
    42: CHECKPOINTS_DIR / "lstm_best.pt",
    123: CHECKPOINTS_DIR / "lstm_unidirectional_seed123_best.pt",
}
predictions = {}

for seed, path in checkpoint_paths.items():
    saved = probabilities[probabilities["SEED"] == seed].sort_values("VALIDATION ROW")
    if (saved["VALIDATION ROW"].tolist() != list(range(len(validation_data)))
            or saved["LABEL"].tolist() != validation_data["label"].tolist()):
        raise ValueError("Saved probabilities do not align with validation rows.")

    checkpoint = torch.load(path, map_location=device, weights_only=True)
    cnn_info = metadata["checkpoint_files"][str(seed)]
    cnn_path = CHECKPOINTS_DIR / cnn_info["textcnn_file"]
    bi_path = CHECKPOINTS_DIR / cnn_info["bilstm_file"]
    if hashlib.sha256(cnn_path.read_bytes()).hexdigest() != cnn_info["textcnn_sha256"]:
        raise ValueError("TextCNN checkpoint does not match the saved probabilities.")
    if hashlib.sha256(bi_path.read_bytes()).hexdigest() != cnn_info["bilstm_sha256"]:
        raise ValueError("BiLSTM checkpoint does not match the saved probabilities.")
    cnn_checkpoint = torch.load(cnn_path, map_location=device, weights_only=True)
    if (checkpoint["vocabulary"] != cnn_checkpoint["vocabulary"]
            or checkpoint["max_length"] != cnn_checkpoint["max_length"]
            or checkpoint["token_pattern"] != TOKEN_PATTERN.pattern
            or checkpoint["token_pattern"] != cnn_checkpoint["token_pattern"]
            or not checkpoint["lowercase"] or not cnn_checkpoint["lowercase"]
            or checkpoint["random_seed"] != seed):
        raise ValueError("Checkpoint sequence processing or seed does not match.")

    dataset = SentimentDataset(validation_data, checkpoint["vocabulary"], checkpoint["max_length"])
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    model = LSTMClassifier(**checkpoint["model_config"]).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    labels = []
    lstm_probabilities = []
    with torch.no_grad():
        for token_ids, lengths, batch_labels in loader:
            logits = model(token_ids.to(device), lengths)
            labels.extend(batch_labels.tolist())
            lstm_probabilities.extend(torch.softmax(logits, dim=1)[:, 1].cpu().tolist())

    if labels != validation_data["label"].tolist():
        raise ValueError("LSTM predictions do not align with validation rows.")
    predictions[seed] = {
        "label": np.array(labels),
        "cnn": saved["TEXTCNN PROBABILITY"].to_numpy(),
        "uni": np.array(lstm_probabilities),
        "bi": saved["BILSTM PROBABILITY"].to_numpy(),
    }
    print(f"Seed {seed}: single-direction LSTM epoch {checkpoint['epoch']}")


In [ ]:
# 分别统计融合相对于相同 CNN 的纠错与新增错误
rows = []
details = {}
for seed, data in predictions.items():
    label = data["label"]
    cnn_prediction = (data["cnn"] > 0.5).astype(int)
    cnn_correct = cnn_prediction == label

    for name, key in [("Unidirectional", "uni"), ("Bidirectional", "bi")]:
        ensemble_prediction = ((data["cnn"] + data[key]) / 2 > 0.5).astype(int)
        ensemble_correct = ensemble_prediction == label
        rows.append({
            "SEED": seed, "LSTM": name,
            **classification_metrics(label, ensemble_prediction),
            "CORRECTED": int((~cnn_correct & ensemble_correct).sum()),
            "NEW ERRORS": int((cnn_correct & ~ensemble_correct).sum()),
            "NET CORRECT": int(ensemble_correct.sum() - cnn_correct.sum()),
        })
        details[(seed, name)] = ensemble_prediction

results = pd.DataFrame(rows)
display(results.round(4))
results.to_csv(TABLES_DIR / "cnn_lstm_direction_error_comparison.csv", index=False)


In [ ]:
# 比较两个融合方案之间不同的预测
rows = []
for seed, data in predictions.items():
    labels = data["label"]
    uni_correct = details[(seed, "Unidirectional")] == labels
    bi_correct = details[(seed, "Bidirectional")] == labels
    rows.append({
        "SEED": seed,
        "BOTH CORRECT": int((uni_correct & bi_correct).sum()),
        "ONLY UNI CORRECT": int((uni_correct & ~bi_correct).sum()),
        "ONLY BI CORRECT": int((~uni_correct & bi_correct).sum()),
        "BOTH WRONG": int((~uni_correct & ~bi_correct).sum()),
    })
direction_summary = pd.DataFrame(rows)
display(direction_summary)
direction_summary.to_csv(TABLES_DIR / "cnn_lstm_direction_pairwise_comparison.csv", index=False)


In [ ]:
# 查看种子 42 两种融合方案判断不同的样本
sample_comparison = validation_data[["text", "label"]].copy()
sample_comparison["CNN"] = (predictions[42]["cnn"] > 0.5).astype(int)
sample_comparison["UNI ENSEMBLE"] = details[(42, "Unidirectional")]
sample_comparison["BI ENSEMBLE"] = details[(42, "Bidirectional")]

uni_correct = sample_comparison["UNI ENSEMBLE"] == sample_comparison["label"]
bi_correct = sample_comparison["BI ENSEMBLE"] == sample_comparison["label"]
for name, mask in [("Only unidirectional ensemble correct", uni_correct & ~bi_correct),
                   ("Only bidirectional ensemble correct", ~uni_correct & bi_correct)]:
    print(name)
    display(sample_comparison[mask].head(3))


## 阶段性判断

种子 42 下，单向融合相对 CNN 纠正 117 条、引入 75 条错误；双向融合纠正 117 条、引入 95 条。种子 123 下，单向融合纠正 140 条、引入 101 条，双向融合纠正 143 条、引入 97 条。

两种等权方案的差异随种子变化：单向在种子 42 更好，双向在种子 123 略好。可以暂将已有的 CNN + 单向 LSTM 等权融合定为主线候选，双向等权融合作为对照；这不是模型普遍优劣的结论。

两个种子都使用相同验证集，且验证集已多次用于模型选择。尚需在固定方案后用未参与选择的数据评估。